<a href="https://colab.research.google.com/github/Aaarkun/rfm-transaction-analytics/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%97%D0%B0%D1%85%D0%B0%D1%80%D0%B5%D0%B2%D0%B8%D1%87_%D0%A4%D0%86%D0%A2_3_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Захаревич Владислав Олегович  
**Група:** 3-12  
**Дата:** 23.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# 2.1 — Дослідження набору даних
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
print('Унікальних користувачів:', n_users)
print('Унікальних фільмів:', n_items)
print('rating: min =', ratings_df['rating'].min(),
      '| max =', ratings_df['rating'].max(),
      '| mean =', round(ratings_df['rating'].mean(), 3))
print('Пропущені значення:\n', ratings_df.isna().sum())
print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())
print('Розрідженість матриці: {:.2%}'.format(1 - len(ratings_df) / (n_users * n_items)))

Унікальних користувачів: 943
Унікальних фільмів: 1682
rating: min = 1.0 | max = 5.0 | mean = 3.53
Пропущені значення:
 user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64
Розрідженість матриці: 93.70%


**Короткі спостереження щодо даних:**

Набір MovieLens 100K містить 100 000 оцінок від 943 користувачів на 1682 фільми за шкалою від 1 до 5 (пропусків немає). Матриця «користувач × фільм» дуже розріджена: заповнено лише близько 6,3% комірок. Оцінки зміщені до високих значень (найчастіші — 4 і 3, середнє близько 3,5).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# 3.1 — Розбиття 80/20
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('train:', train_df.shape, '| test:', test_df.shape)

train: (80000, 4) | test: (20000, 4)


In [6]:
# 3.2 — Об'єкти для Surprise
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('rating_scale:', reader.rating_scale)
print('train raw_ratings:', len(train_data.raw_ratings), '| testset:', len(testset))

rating_scale: (1, 5)
train raw_ratings: 80000 | testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD `n_factors` задає розмірність латентного простору (більше факторів — вища виразність, але ризик перенавчання), `lr_all` — швидкість градієнтного спуску, `reg_all` — L2-регуляризацію, що стримує перенавчання на розріджених даних. Для KNNBasic `k` — максимальна кількість сусідів, `min_k` — мінімум сусідів для прогнозу (інакше використовується середнє), `sim_options.name` — міра схожості (cosine / pearson), `user_based` — порівняння користувачів або фільмів. Початкову сітку із шаблону залишено без змін: вона невелика, швидко рахується в Colab і покриває основні режими обох алгоритмів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# 5.1 — GridSearchCV для SVD
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('SVD готово')

SVD готово


In [9]:
# 5.2 — GridSearchCV для KNNBasic
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('KNNBasic готово')

KNNBasic готово


In [10]:
# 5.3 — Найкращі результати CV
def cv_summary(gs, name):
    idx = gs.best_index['rmse']
    return {
        'Model': name,
        'CV RMSE': gs.best_score['rmse'],
        'CV MAE': gs.cv_results['mean_test_mae'][idx],
        'Best params (RMSE)': gs.best_params['rmse'],
        'Best MAE (за MAE)': gs.best_score['mae'],
    }

cv_table = pd.DataFrame([
    cv_summary(grid_search_svd, 'SVD'),
    cv_summary(grid_search_knn, 'KNNBasic'),
])
display(cv_table)
print('SVD best params (rmse):     ', grid_search_svd.best_params['rmse'])
print('KNNBasic best params (rmse):', grid_search_knn.best_params['rmse'])

,Model,CV RMSE,CV MAE,Best params (RMSE),Best MAE (за MAE)
0,SVD,0.942774,0.743419,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0...",0.743419
1,KNNBasic,1.028362,0.815442,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ...",0.815442


SVD best params (rmse):      {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
KNNBasic best params (rmse): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:** таблиця та висновок формуються автоматично наступною коміркою.

In [11]:
from IPython.display import Markdown
rows = []
for gs, name in [(grid_search_svd, 'SVD'), (grid_search_knn, 'KNNBasic')]:
    i = gs.best_index['rmse']
    rows.append(f"| {name} | {gs.best_score['rmse']:.4f} | {gs.cv_results['mean_test_mae'][i]:.4f} | `{gs.best_params['rmse']}` |")
better_cv = 'SVD' if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse'] else 'KNNBasic'
display(Markdown(
    "| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |\n|:--|--:|--:|:--|\n"
    + "\n".join(rows)
    + f"\n\nЗа кросвалідацією кращою є модель **{better_cv}** (нижчий RMSE)."
))

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9428 | 0.7434 | `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}` |
| KNNBasic | 1.0284 | 0.8154 | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

За кросвалідацією кращою є модель **SVD** (нижчий RMSE).

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# 6.1 — Найкращі параметри за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# 6.2 — Повна навчальна вибірка
trainset = train_data.build_full_trainset()
print('Користувачів у trainset:', trainset.n_users, '| фільмів:', trainset.n_items,
      '| оцінок:', trainset.n_ratings)

# 6.3 — Навчання остаточних моделей
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params, verbose=False)
best_knn.fit(trainset)
print('Моделі навчено')

Користувачів у trainset: 943 | фільмів: 1653 | оцінок: 80000
Моделі навчено


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# 7.1 — Передбачення на тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# 7.2 — Метрики
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae  = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae  = accuracy.mae(knn_predictions, verbose=False)
print(f'SVD:      RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNNBasic: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

SVD:      RMSE=0.9192, MAE=0.7234
KNNBasic: RMSE=1.0120, MAE=0.7984


In [14]:
# 7.3 — Порівняльна таблиця
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae],
}).sort_values('RMSE').reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів та порівняння з CV:** формуються автоматично наступною коміркою.

In [15]:
t = results.set_index('Model')
cv = {'SVD': grid_search_svd.best_score['rmse'], 'KNNBasic': grid_search_knn.best_score['rmse']}
lines = [f"| {m} | {t.loc[m,'RMSE']:.4f} | {t.loc[m,'MAE']:.4f} |" for m in ['SVD', 'KNNBasic']]
diffs = ', '.join(f"{m}: {abs(t.loc[m,'RMSE']-cv[m]):.4f}" for m in ['SVD', 'KNNBasic'])
display(Markdown(
    "| Модель | Test RMSE | Test MAE |\n|:--|--:|--:|\n" + "\n".join(lines)
    + f"\n\n**Порівняння з CV:** абсолютна різниця RMSE між CV і тестом — {diffs}. "
      "Невелика різниця означає, що моделі не перенавчені, а оцінка якості стабільна."
))

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** абсолютна різниця RMSE між CV і тестом — SVD: 0.0236, KNNBasic: 0.0164. Невелика різниця означає, що моделі не перенавчені, а оцінка якості стабільна.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [16]:
# Початковий ID
user_id = '196'

# 8.1 — Найкраща за CV RMSE модель
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model, best_name = best_svd, 'SVD'
else:
    best_model, best_name = best_knn, 'KNNBasic'
print('Обрана модель за CV RMSE:', best_name)

# 8.2 — Чи є користувач у trainset
try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    inner_uid = None
    print(f'Користувача {user_id} немає у навчальній вибірці — персональні рекомендації неможливі (cold start).')

if inner_uid is not None:
    # 8.3 — Фільми, які користувач уже оцінив у train
    rated_items = {trainset.to_raw_iid(iid) for iid, _ in trainset.ur[inner_uid]}
    all_items = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]
    unrated_items = [iid for iid in all_items if iid not in rated_items]
    print('Оцінено фільмів:', len(rated_items), '| кандидатів:', len(unrated_items))

    # 8.4 — Прогноз та топ-10
    preds = [(iid, best_model.predict(uid=user_id, iid=iid).est) for iid in unrated_items]
    top10 = (pd.DataFrame(preds, columns=['item_id', 'predicted_rating'])
               .sort_values('predicted_rating', ascending=False)
               .head(10).reset_index(drop=True))
    display(top10)

Обрана модель за CV RMSE: SVD
Оцінено фільмів: 29 | кандидатів: 1624


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [17]:
# Додаткове завдання: власна рекомендаційна модель з нуля
# Біасована матрична факторизація (FunkSVD) на NumPy: r_ui ≈ mu + b_u + b_i + p_u · q_i,
# оптимізація стохастичним градієнтним спуском із L2-регуляризацією.
class MyFunkSVD:
    def __init__(self, n_factors=50, lr=0.01, reg=0.05, n_epochs=25, random_state=42):
        self.n_factors, self.lr, self.reg = n_factors, lr, reg
        self.n_epochs, self.rng = n_epochs, np.random.RandomState(random_state)

    def fit(self, df):
        self.u_map = {u: i for i, u in enumerate(df['user_id'].unique())}
        self.i_map = {m: i for i, m in enumerate(df['item_id'].unique())}
        u = df['user_id'].map(self.u_map).to_numpy()
        i = df['item_id'].map(self.i_map).to_numpy()
        r = df['rating'].to_numpy(dtype=float)
        self.mu = r.mean()
        self.bu = np.zeros(len(self.u_map)); self.bi = np.zeros(len(self.i_map))
        self.P = self.rng.normal(0, 0.1, (len(self.u_map), self.n_factors))
        self.Q = self.rng.normal(0, 0.1, (len(self.i_map), self.n_factors))
        for epoch in range(self.n_epochs):
            for k in self.rng.permutation(len(r)):
                a, b = u[k], i[k]
                err = r[k] - (self.mu + self.bu[a] + self.bi[b] + self.P[a] @ self.Q[b])
                self.bu[a] += self.lr * (err - self.reg * self.bu[a])
                self.bi[b] += self.lr * (err - self.reg * self.bi[b])
                pa = self.P[a].copy()
                self.P[a] += self.lr * (err * self.Q[b] - self.reg * pa)
                self.Q[b] += self.lr * (err * pa - self.reg * self.Q[b])
        return self

    def predict(self, uid, iid):
        est = self.mu
        ui, ii = self.u_map.get(uid), self.i_map.get(iid)
        if ui is not None: est += self.bu[ui]
        if ii is not None: est += self.bi[ii]
        if ui is not None and ii is not None: est += self.P[ui] @ self.Q[ii]
        return float(np.clip(est, 1, 5))

my_model = MyFunkSVD(n_factors=50, lr=0.01, reg=0.05, n_epochs=25).fit(train_df)
my_preds = np.array([my_model.predict(u, i) for u, i, _ in testset])
true = np.array([r for _, _, r in testset])
my_rmse = np.sqrt(np.mean((my_preds - true) ** 2)); my_mae = np.mean(np.abs(my_preds - true))
print(f'MyFunkSVD: RMSE={my_rmse:.4f}, MAE={my_mae:.4f}')
display(pd.concat([results, pd.DataFrame({'Model': ['MyFunkSVD (з нуля)'], 'RMSE': [my_rmse], 'MAE': [my_mae]})],
                  ignore_index=True).sort_values('RMSE').reset_index(drop=True))

MyFunkSVD: RMSE=0.9212, MAE=0.7238


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,MyFunkSVD (з нуля),0.921178,0.723850
2,KNNBasic,1.011953,0.798421


In [18]:
# Автоматично сформовані числові висновки
better_cv = 'SVD' if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse'] else 'KNNBasic'
display(Markdown(f'''
**Висновок.** За кросвалідацією (RMSE) кращою обрано модель **{better_cv}**. Найкращі гіперпараметри: SVD — `{best_svd_params}`, KNNBasic — `{best_knn_params}`. На незалежній тестовій вибірці SVD має RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}, а KNNBasic — RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}. Тестові оцінки близькі до CV, тому перенавчання немає. Обмеження: нові користувачі та фільми (cold start) без історії оцінок отримують лише глобально-базові прогнози, а матриця дуже розріджена. Власна реалізація FunkSVD дала RMSE={my_rmse:.4f}, MAE={my_mae:.4f}, що порівнянно з бібліотечним SVD.
'''))


**Висновок.** За кросвалідацією (RMSE) кращою обрано модель **SVD**. Найкращі гіперпараметри: SVD — `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}`, KNNBasic — `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}`. На незалежній тестовій вибірці SVD має RMSE=0.9192, MAE=0.7234, а KNNBasic — RMSE=1.0120, MAE=0.7984. Тестові оцінки близькі до CV, тому перенавчання немає. Обмеження: нові користувачі та фільми (cold start) без історії оцінок отримують лише глобально-базові прогнози, а матриця дуже розріджена. Власна реалізація FunkSVD дала RMSE=0.9212, MAE=0.7238, що порівнянно з бібліотечним SVD.
